# FindFire phase 2: MTG early alerts matched to VIIRS zones

Adds a geostationary early-alert layer on top of the VIIRS zones, following the project plan (section 7):
alerts are matched to VIIRS inside an N x N pixel window and a time window, and the **lead time** (first VIIRS detection
minus first alert) is measured per zone.

**How to run:** keep this file next to `FindFire_zones_tracking_replay.ipynb` and run all cells. The first cell runs that notebook
for you if `df`, `zones` and `EVENT` do not exist yet. (Or copy these cells to the end of that notebook.)

By default this is a **dry run on fake detections**. Set `MTG_CSV` in the last section to use real ones.

In [ ]:
# Needs df, zones and EVENT from FindFire_zones_tracking_replay.ipynb
if not {"df", "zones", "EVENT"} <= set(globals()):
    get_ipython().run_line_magic("run", "-i FindFire_zones_tracking_replay.ipynb")
print(len(df), "VIIRS hotspots,", sum(z["merged_into"] is None for z in zones.values()), "zones")

## 1. Settings and helpers
Edit `Params` to change the window sizes. Defaults follow published practice: 3 x 3 pixel window, 12 h time window.

In [ ]:
"""FindFire phase 2: geostationary (MTG) early alerts matched against VIIRS zones.

Follows the practice described in the project plan (section 7):
  * VIIRS (high / nominal confidence) is the reference for geostationary detections.
  * Matching uses a spatial buffer of N x N geostationary pixels (default 3 x 3), not a fixed distance.
  * A time window (default 12 h) lets VIIRS confirm an alert at its next pass.
  * An alert is "persistent" when it repeats (>= 2 detections within 12 h of each other).
  * Lead time = first VIIRS detection minus first alert. Report a median and a range, never one number.
  * Frame-level precision and recall by fire size at VIIRS passes, with bootstrap confidence intervals.

Inputs
  det    DataFrame of geostationary detections: latitude, longitude, time (UTC), frp, pixel_km, quality
  viirs  the project's hotspot DataFrame: latitude, longitude, acq_dt (UTC), scan, track (km), frp
  zones  the dict returned by run_tracking(): {zone_id: {"idx": set, "first": ts, "last": ts, "merged_into": id|None}}

Everything here is tuned for tens-of-km distances (flat-earth offsets), which is what the matching needs.
"""
from __future__ import annotations

import json
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

EARTH_R_KM = 6371.0088
KM_PER_DEG = np.pi / 180 * EARTH_R_KM


@dataclass(frozen=True)
class Params:
    pixel_km: float = 2.0         # geostationary pixel size if the file has none (nominal; larger over Algeria)
    window_pixels: int = 3        # spatial buffer: N x N geostationary pixels (published practice: 3 x 3)
    window_h: float = 12.0        # VIIRS may confirm an alert up to this long after the first detection
    back_h: float = 12.0          # ... or this long before it (VIIRS may already have seen the fire)
    gap_h: float = 12.0           # detections of the same place within this gap form one alert
    link_pixels: float = 1.5      # detections closer than this many pixels belong to the same alert
    persistence_min: int = 2      # detections needed for an alert to count as persistent
    pass_tol_min: float = 10.0    # time tolerance around a VIIRS pass for frame-level metrics


def _ts(x) -> pd.Timestamp:
    t = pd.Timestamp(x)
    return t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")


def _ns(x) -> np.datetime64:
    return _ts(x).tz_localize(None).to_datetime64()


def _ns_array(series: pd.Series) -> np.ndarray:
    return pd.to_datetime(series, utc=True).dt.tz_localize(None).to_numpy("datetime64[ns]")


def _offsets_km(lat0, lon0, lat, lon):
    """East / north offsets (km) of (lat, lon) from (lat0, lon0)."""
    lat0 = np.asarray(lat0, float)
    dx = (np.asarray(lon, float) - np.asarray(lon0, float)) * KM_PER_DEG * np.cos(np.radians(lat0))
    dy = (np.asarray(lat, float) - lat0) * KM_PER_DEG
    return dx, dy


def _viirs_size_km(viirs: pd.DataFrame) -> np.ndarray:
    if {"scan", "track"} <= set(viirs.columns):
        return np.maximum(viirs["scan"].to_numpy(float), viirs["track"].to_numpy(float))
    return np.full(len(viirs), 0.375)


def hotspot_zone_map(zones: dict) -> dict:
    """{hotspot index: zone_id} for the zones that still exist (merged-away zones are resolved)."""
    return {i: zid for zid, z in zones.items() if z["merged_into"] is None for i in z["idx"]}

## 2. Load geostationary detections
Reads any CSV (FIRMS-style `acq_date` + `acq_time`, or a `time` column).

In [ ]:
def load_detections_csv(path, **cols) -> pd.DataFrame:
    """Read geostationary detections from a CSV and return the standard columns.

    Override column names if needed, e.g. load_detections_csv("mtg.csv", time="datetime", frp="frp_mw").
    A time column can also be given as FIRMS-style acq_date + acq_time (HHMM)."""
    names = dict(latitude="latitude", longitude="longitude", time="time", frp="frp",
                 pixel_km="pixel_km", quality="quality")
    names.update(cols)
    raw = pd.read_csv(path)
    out = pd.DataFrame({"latitude": raw[names["latitude"]], "longitude": raw[names["longitude"]]})
    if names["time"] in raw:
        out["time"] = pd.to_datetime(raw[names["time"]], utc=True)
    elif {"acq_date", "acq_time"} <= set(raw):
        hhmm = raw["acq_time"].astype(int).astype(str).str.zfill(4)
        out["time"] = pd.to_datetime(raw["acq_date"].astype(str) + " " + hhmm, format="%Y-%m-%d %H%M", utc=True)
    else:
        raise ValueError("no time column found: pass time='<column name>'")
    out["frp"] = raw[names["frp"]] if names["frp"] in raw else np.nan
    out["pixel_km"] = raw[names["pixel_km"]] if names["pixel_km"] in raw else np.nan
    out["quality"] = raw[names["quality"]] if names["quality"] in raw else ""
    return out.sort_values("time").reset_index(drop=True)

## 3. Detections to alerts
Repeated detections of the same place become one alert. An alert with 2 or more detections is *persistent*.

In [ ]:
ALERT_COLS = ["alert_id", "source", "first_detected", "last_detected", "latitude", "longitude",
              "pixel_km", "frp_mw", "n_detections", "persistent"]


def build_alerts(det: pd.DataFrame, p: Params = Params()) -> pd.DataFrame:
    """Group repeated geostationary detections of the same place into alerts (one per fire)."""
    if det.empty:
        return pd.DataFrame(columns=ALERT_COLS)
    det = det.sort_values("time").reset_index(drop=True)
    lat, lon = det.latitude.to_numpy(float), det.longitude.to_numpy(float)
    pk = det.pixel_km.fillna(p.pixel_km).to_numpy(float)
    frp = det.frp.fillna(0).to_numpy(float)
    times = pd.to_datetime(det.time, utc=True)
    gap = pd.Timedelta(hours=p.gap_h)
    alerts, active = [], []
    for i in range(len(det)):
        t = times[i]
        active = [a for a in active if alerts[a]["last"] >= t - gap]
        best, best_d = None, np.inf
        for a in active:
            A = alerts[a]
            dx, dy = _offsets_km(A["lat"], A["lon"], lat[i], lon[i])           # distance to the alert centre
            ex, ey = _offsets_km(A["llat"], A["llon"], lat[i], lon[i])         # ... and to its latest detection,
            d = float(min(np.hypot(dx, dy), np.hypot(ex, ey)))                # so a moving fire front stays one alert
            if d <= p.link_pixels * max(pk[i], A["pixel_km"]) and d < best_d:
                best, best_d = a, d
        if best is None:
            alerts.append(dict(lat=lat[i], lon=lon[i], llat=lat[i], llon=lon[i], first=t, last=t, n=1,
                               frp=frp[i], pixel_km=pk[i]))
            active.append(len(alerts) - 1)
        else:
            A = alerts[best]
            A["lat"] += (lat[i] - A["lat"]) / (A["n"] + 1)
            A["lon"] += (lon[i] - A["lon"]) / (A["n"] + 1)
            A["n"] += 1; A["last"] = t; A["llat"], A["llon"] = lat[i], lon[i]
            A["frp"] = max(A["frp"], frp[i]); A["pixel_km"] = max(A["pixel_km"], pk[i])
    return pd.DataFrame(dict(
        alert_id=np.arange(1, len(alerts) + 1), source="MTG",
        first_detected=[a["first"] for a in alerts], last_detected=[a["last"] for a in alerts],
        latitude=[a["lat"] for a in alerts], longitude=[a["lon"] for a in alerts],
        pixel_km=[a["pixel_km"] for a in alerts], frp_mw=[a["frp"] for a in alerts],
        n_detections=[a["n"] for a in alerts],
        persistent=[a["n"] >= p.persistence_min for a in alerts]))

## 4. Match alerts to VIIRS
confirmed / pending / not confirmed, with lead time in minutes (positive = MTG was earlier).

In [ ]:
def match_alerts(alerts: pd.DataFrame, viirs: pd.DataFrame, hotspot_zone: dict | None = None,
                 p: Params = Params(), as_of=None) -> pd.DataFrame:
    """Match each alert against VIIRS detections inside an N x N pixel window and a time window.

    status       confirmed      a VIIRS detection is inside the window (space and time)
                 pending        no match yet, and the time window is still open (as of `as_of`)
                 not_confirmed  no match and the time window has closed
    lead_time_min  first VIIRS detection minus first alert, in minutes (positive = MTG was earlier)
    zone_id        zone of the earliest matching VIIRS detection that belongs to a zone

    Note: FIRMS gives detections, not the list of overpasses, so a closed window is taken to mean that
    VIIRS had its chance. Pass real overpass times to frame_metrics() for a stricter test."""
    hotspot_zone = hotspot_zone or {}
    out = alerts.copy()
    for c, v in dict(status="pending", n_viirs=0, lead_time_min=np.nan).items():
        out[c] = v
    out["confirmed_at"] = pd.Series(pd.NaT, index=out.index, dtype="datetime64[ns, UTC]")
    out["zone_id"] = pd.array([pd.NA] * len(out), dtype="Int64")
    if out.empty or viirs.empty:          # nothing to match, or no VIIRS data: alerts stay pending
        return out
    v = viirs.sort_values("acq_dt")
    vt = _ns_array(v.acq_dt)
    lat, lon = v.latitude.to_numpy(float), v.longitude.to_numpy(float)
    size, idx = _viirs_size_km(v), v.index.to_numpy()
    as_of = _ts(as_of if as_of is not None else v.acq_dt.max())
    for row in out.itertuples():
        t0 = _ts(row.first_detected)
        lo = np.searchsorted(vt, _ns(t0 - pd.Timedelta(hours=p.back_h)), "left")
        hi = np.searchsorted(vt, _ns(t0 + pd.Timedelta(hours=p.window_h)), "right")
        hit = np.array([], dtype=int)
        if hi > lo:
            dx, dy = _offsets_km(row.latitude, row.longitude, lat[lo:hi], lon[lo:hi])
            half = 0.5 * p.window_pixels * row.pixel_km + 0.5 * size[lo:hi]
            hit = np.nonzero((np.abs(dx) <= half) & (np.abs(dy) <= half))[0] + lo
        if len(hit):
            first = hit[0]                                  # sorted by time, so the earliest match
            out.at[row.Index, "status"] = "confirmed"
            out.at[row.Index, "n_viirs"] = len(hit)
            out.at[row.Index, "confirmed_at"] = pd.Timestamp(vt[first], tz="UTC")
            out.at[row.Index, "lead_time_min"] = (vt[first] - _ns(t0)) / np.timedelta64(1, "m")
            zone = next((hotspot_zone[idx[j]] for j in hit if idx[j] in hotspot_zone), None)
            if zone is not None:
                out.at[row.Index, "zone_id"] = zone
        elif as_of >= t0 + pd.Timedelta(hours=p.window_h):
            out.at[row.Index, "status"] = "not_confirmed"
    return out

## 5. Lead time per zone

In [ ]:
def zone_lead_times(zones: dict, alerts: pd.DataFrame) -> pd.DataFrame:
    """For each live zone: first VIIRS detection, first linked MTG alert and the lead time between them."""
    conf = alerts[alerts.status == "confirmed"]
    rows = []
    for zid, z in zones.items():
        if z["merged_into"] is not None:
            continue
        mine = conf[conf.zone_id.eq(zid).fillna(False)]
        first_alert = mine.first_detected.min() if len(mine) else pd.NaT
        lead = (_ts(z["first"]) - _ts(first_alert)).total_seconds() / 60 if len(mine) else np.nan
        rows.append(dict(zone_id=zid, first_viirs=z["first"], first_alert=first_alert,
                         lead_time_min=lead, n_alerts=len(mine), has_alert=len(mine) > 0))
    return pd.DataFrame(rows, columns=["zone_id", "first_viirs", "first_alert", "lead_time_min", "n_alerts", "has_alert"])

## 6. Statistics
Bootstrap confidence intervals (1500 repetitions), headline numbers and frame-level precision / recall by fire size.

In [ ]:
def bootstrap_prop(successes: int, n: int, reps: int = 1500, seed: int = 0):
    """(proportion, 2.5 %, 97.5 %) by bootstrap (1500 repetitions, as in the published protocol)."""
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = successes / n
    draws = np.random.default_rng(seed).binomial(n, p, size=reps) / n
    return (p, *np.percentile(draws, [2.5, 97.5]))


def bootstrap_median(values, reps: int = 1500, seed: int = 0):
    v = np.asarray(values, float); v = v[~np.isnan(v)]
    if len(v) == 0:
        return (np.nan, np.nan, np.nan)
    meds = np.median(np.random.default_rng(seed).choice(v, (reps, len(v))), axis=1)
    return (float(np.median(v)), *np.percentile(meds, [2.5, 97.5]))


def summarize(alerts: pd.DataFrame, zone_lt: pd.DataFrame) -> dict:
    """Headline numbers for the report. Lead time is a distribution: use the median and the range."""
    out = {"alerts": len(alerts), **{f"status_{k}": int(v) for k, v in alerts.status.value_counts().items()}}
    closed = alerts[alerts.status != "pending"]
    out["confirmation_rate (all closed alerts)"] = bootstrap_prop(int((closed.status == "confirmed").sum()), len(closed))
    pers = closed[closed.persistent]
    out["confirmation_rate (persistent only)"] = bootstrap_prop(int((pers.status == "confirmed").sum()), len(pers))
    out["zones"] = len(zone_lt)
    out["zones_with_alert"] = bootstrap_prop(int(zone_lt.has_alert.sum()), len(zone_lt))
    lt = zone_lt.lead_time_min.dropna()
    out["zones_alerted_before_viirs"] = int((lt > 0).sum())
    out["lead_time_median_min (95% CI)"] = bootstrap_median(lt)
    out["lead_time_range_min"] = (float(lt.min()), float(lt.max())) if len(lt) else (np.nan, np.nan)
    return out


def frame_metrics(det: pd.DataFrame, viirs: pd.DataFrame, p: Params = Params(), pass_times=None,
                  reps: int = 1500, seed: int = 0) -> pd.DataFrame:
    """Frame-level comparison at VIIRS passes (Hall 2019, Vanunu 2026).

    precision  share of geostationary detections near a pass that have a VIIRS detection in their window
    recall     share of VIIRS fire clusters (about one geostationary pixel across) that have a geostationary
               detection in range, reported by fire size (number of VIIRS pixels in the cluster)
    pass_times optional list of (start, end) overpass times; by default passes are inferred from VIIRS
               detection times (gap > 30 min starts a new pass). Passing real overpass times avoids
               ignoring passes where VIIRS saw nothing, which would inflate precision."""
    v = viirs.sort_values("acq_dt")
    if pass_times is None:
        pid = (v.acq_dt.diff() > pd.Timedelta(minutes=30)).cumsum()
        pass_times = [(g.acq_dt.min(), g.acq_dt.max()) for _, g in v.groupby(pid.values)]
    tol = pd.Timedelta(minutes=p.pass_tol_min)
    det_t = pd.to_datetime(det.time, utc=True)
    prec_flags, sizes, found = [], [], []
    for start, end in pass_times:
        D = det[(det_t >= _ts(start) - tol) & (det_t <= _ts(end) + tol)]
        V = v[(v.acq_dt >= _ts(start)) & (v.acq_dt <= _ts(end))]
        if D.empty and V.empty:
            continue
        if V.empty:
            prec_flags += [False] * len(D); continue
        if D.empty:
            M = np.zeros((0, len(V)), bool)
        else:
            pxD = D.pixel_km.fillna(p.pixel_km).to_numpy(float)
            dx, dy = _offsets_km(D.latitude.to_numpy()[:, None], D.longitude.to_numpy()[:, None],
                                 V.latitude.to_numpy()[None, :], V.longitude.to_numpy()[None, :])
            half = 0.5 * p.window_pixels * pxD[:, None] + 0.5 * _viirs_size_km(V)[None, :]
            M = (np.abs(dx) <= half) & (np.abs(dy) <= half)
        prec_flags += list(M.any(axis=1))
        lab = DBSCAN(eps=p.pixel_km / EARTH_R_KM, min_samples=1, metric="haversine", algorithm="ball_tree"
                     ).fit_predict(np.radians(np.c_[V.latitude.to_numpy(), V.longitude.to_numpy()]))
        hit_v = M.any(axis=0)
        for k in np.unique(lab):
            sel = lab == k
            sizes.append(int(sel.sum())); found.append(bool(hit_v[sel].any()))
    rows = []
    tp = int(sum(prec_flags)); rows.append(("precision (detections with VIIRS nearby)", len(prec_flags), *bootstrap_prop(tp, len(prec_flags), reps, seed)))
    sizes, found = np.array(sizes, int), np.array(found, bool)
    for label, lo, hi in [("recall, fire size 1 VIIRS pixel", 1, 1), ("recall, fire size 2-3", 2, 3),
                          ("recall, fire size 4-6", 4, 6), ("recall, fire size 7+", 7, 10**9),
                          ("recall, all fire clusters", 1, 10**9)]:
        m = (sizes >= lo) & (sizes <= hi)
        rows.append((label, int(m.sum()), *bootstrap_prop(int(found[m].sum()), int(m.sum()), reps, seed)))
    return pd.DataFrame(rows, columns=["metric", "n", "value", "ci_low", "ci_high"])

## 7. Export and dry-run data
`make_synthetic_mtg` creates fake detections for testing the code path only.

In [ ]:
def export_alerts_json(alerts: pd.DataFrame, path) -> None:
    """Write alerts in the data-contract format (data/<event>/alerts.json)."""
    def iso(t):
        return None if pd.isna(t) else _ts(t).strftime("%Y-%m-%dT%H:%M:%SZ")

    def num(x, nd=2):
        return None if pd.isna(x) else round(float(x), nd)
    recs = [dict(alert_id=int(r.alert_id), source=r.source, first_detected=iso(r.first_detected),
                 last_detected=iso(r.last_detected), lat=round(float(r.latitude), 5), lon=round(float(r.longitude), 5),
                 pixel_km=num(r.pixel_km), frp_mw=num(r.frp_mw, 1), n_detections=int(r.n_detections),
                 persistent=bool(r.persistent), status=r.status,
                 zone_id=None if pd.isna(r.zone_id) else int(r.zone_id),
                 confirmed_at=iso(r.confirmed_at), lead_time_min=num(r.lead_time_min, 1))
            for r in alerts.itertuples()]
    with open(path, "w") as f:
        json.dump(recs, f)


def make_synthetic_mtg(viirs: pd.DataFrame, zones: dict, seed: int = 0, miss_every: int = 4,
                       n_false: int = 6) -> pd.DataFrame:
    """DRY RUN ONLY: fake geostationary detections built from the VIIRS zones, to test the code path.

    Each zone is seen 0.5 to 5 h before VIIRS first sees it (every 10 min, 70 % of frames), every
    `miss_every`-th zone is never seen, and a few isolated false alarms are added.
    Only the alert, confirmation and lead-time numbers mean anything on this data: detections stop a day
    after VIIRS first sees a zone and cluster at its centre, so frame_metrics() recall is not meaningful."""
    rng = np.random.default_rng(seed)
    rows = []
    live = [(zid, z) for zid, z in zones.items() if z["merged_into"] is None]
    for k, (zid, z) in enumerate(live):
        if (k + 1) % miss_every == 0:
            continue
        sub = viirs.loc[sorted(z["idx"])]
        lead_h = rng.uniform(0.5, 5.0)
        start = _ts(z["first"]) - pd.Timedelta(hours=lead_h)
        for f in range(int((lead_h + 24) * 6)):
            if rng.random() < 0.7:
                rows.append(dict(latitude=sub.latitude.mean() + rng.normal(0, 0.01),
                                 longitude=sub.longitude.mean() + rng.normal(0, 0.012),
                                 time=start + pd.Timedelta(minutes=10 * f), frp=float(rng.lognormal(3, 0.5)),
                                 pixel_km=2.0, quality="good"))
    t0, t1 = viirs.acq_dt.min(), viirs.acq_dt.max()
    for _ in range(n_false):
        rows.append(dict(latitude=rng.uniform(viirs.latitude.min(), viirs.latitude.max()),
                         longitude=rng.uniform(viirs.longitude.min(), viirs.longitude.max()),
                         time=t0 + (t1 - t0) * rng.random(), frp=float(rng.lognormal(2, 0.4)),
                         pixel_km=2.0, quality="good"))
    return pd.DataFrame(rows).sort_values("time").reset_index(drop=True)

## 8. Quick tests (known answers)
The same checks as `test_findfire_alerts.py`, run inside the notebook.

In [ ]:
import sys, types, inspect, tempfile, pathlib
fa = sys.modules["__main__"]                      # the functions above live in this notebook

class _Approx:                                    # tiny stand-in for pytest.approx
    def __init__(self, x): self.x = x
    def __eq__(self, other): return abs(other - self.x) <= 1e-6 * max(1.0, abs(self.x))
pytest = types.SimpleNamespace(approx=_Approx)

import numpy as np
import pandas as pd


T0 = pd.Timestamp("2025-08-10 10:00", tz="UTC")
KM_LON = fa.KM_PER_DEG * np.cos(np.radians(36.0))     # km per degree of longitude at 36 N
KM_LAT = fa.KM_PER_DEG


def mtg(rows):
    """rows: (minutes after T0, east km, north km) offsets from a base point at 36 N, 4 E."""
    return pd.DataFrame([dict(latitude=36.0 + n / KM_LAT, longitude=4.0 + e / KM_LON, time=T0 + pd.Timedelta(minutes=m),
                              frp=10.0, pixel_km=2.0, quality="good") for m, e, n in rows])


def viirs(rows):
    """rows: (minutes after T0, east km, north km)."""
    return pd.DataFrame([dict(latitude=36.0 + n / KM_LAT, longitude=4.0 + e / KM_LON, acq_dt=T0 + pd.Timedelta(minutes=m),
                              scan=0.4, track=0.4, frp=5.0) for m, e, n in rows])


# ------------------------------------------------------------------ build_alerts
def test_repeated_detections_form_one_persistent_alert():
    a = fa.build_alerts(mtg([(0, 0, 0), (10, 0.5, 0), (20, 0, 0.5)]))
    assert len(a) == 1 and a.n_detections[0] == 3 and bool(a.persistent[0])


def test_moving_front_stays_one_alert():
    a = fa.build_alerts(mtg([(0, 0, 0), (10, 2.5, 0), (20, 5.0, 0), (30, 7.5, 0)]))   # drifts 2.5 km per frame
    assert len(a) == 1 and a.n_detections[0] == 4


def test_far_detections_are_separate_alerts():
    a = fa.build_alerts(mtg([(0, 0, 0), (10, 20, 0)]))
    assert len(a) == 2


def test_time_gap_splits_alerts_and_single_detection_is_not_persistent():
    a = fa.build_alerts(mtg([(0, 0, 0), (13 * 60, 0, 0)]))          # 13 h apart, same place
    assert len(a) == 2 and not a.persistent.any()


# ------------------------------------------------------------------ match_alerts
def test_confirmed_with_correct_lead_time():
    alerts = fa.build_alerts(mtg([(0, 0, 0), (10, 0, 0)]))
    out = fa.match_alerts(alerts, viirs([(180, 1.0, 1.0)]))           # VIIRS 3 h later, 1.4 km away
    assert out.status[0] == "confirmed" and out.lead_time_min[0] == pytest.approx(180)


def test_not_confirmed_when_viirs_is_far_and_window_closed():
    alerts = fa.build_alerts(mtg([(0, 0, 0), (10, 0, 0)]))
    out = fa.match_alerts(alerts, viirs([(180, 30.0, 0), (14 * 60, 30.0, 0)]))   # 30 km away; data runs past 12 h
    assert out.status[0] == "not_confirmed"


def test_pending_while_window_is_open():
    alerts = fa.build_alerts(mtg([(0, 0, 0), (10, 0, 0)]))
    out = fa.match_alerts(alerts, viirs([(120, 30.0, 0)]))            # data only 2 h after the alert
    assert out.status[0] == "pending"


def test_window_size_changes_the_match():
    alerts = fa.build_alerts(mtg([(0, 0, 0), (10, 0, 0)]))
    v = viirs([(180, 4.0, 0)])                                         # 4 km east: outside 3 x 3 (3.2 km), inside 5 x 5 (5.2 km)
    assert fa.match_alerts(alerts, v, p=fa.Params(window_pixels=3), as_of=T0 + pd.Timedelta(hours=20)).status[0] == "not_confirmed"
    assert fa.match_alerts(alerts, v, p=fa.Params(window_pixels=5)).status[0] == "confirmed"


def test_viirs_before_the_alert_gives_negative_lead():
    alerts = fa.build_alerts(mtg([(120, 0, 0), (130, 0, 0)]))
    out = fa.match_alerts(alerts, viirs([(30, 0, 0)]))                 # VIIRS saw it 90 min earlier
    assert out.status[0] == "confirmed" and out.lead_time_min[0] == pytest.approx(-90)


def test_time_window_limits_confirmation():
    alerts = fa.build_alerts(mtg([(0, 0, 0), (10, 0, 0)]))
    out = fa.match_alerts(alerts, viirs([(13 * 60, 0, 0)]))           # 13 h later: outside the 12 h window
    assert out.status[0] == "not_confirmed"


def test_no_viirs_data_leaves_alerts_pending():
    alerts = fa.build_alerts(mtg([(0, 0, 0)]))
    out = fa.match_alerts(alerts, viirs([]).reindex(columns=["latitude", "longitude", "acq_dt", "scan", "track", "frp"]))
    assert out.status[0] == "pending"


# ------------------------------------------------------------------ zones
def test_zone_lead_time_and_missed_zone():
    v = viirs([(180, 0, 0), (185, 0.3, 0), (200, 50, 50)])
    zones = {1: dict(idx={0, 1}, first=v.acq_dt[0], last=v.acq_dt[1], merged_into=None),
             2: dict(idx={2}, first=v.acq_dt[2], last=v.acq_dt[2], merged_into=None),
             3: dict(idx={0}, first=v.acq_dt[0], last=v.acq_dt[0], merged_into=1)}      # merged away: ignored
    alerts = fa.match_alerts(fa.build_alerts(mtg([(0, 0, 0), (10, 0, 0)])), v, fa.hotspot_zone_map(zones))
    zl = fa.zone_lead_times(zones, alerts).set_index("zone_id")
    assert list(zl.index) == [1, 2]
    assert zl.loc[1, "lead_time_min"] == pytest.approx(180) and bool(zl.loc[1, "has_alert"])
    assert not zl.loc[2, "has_alert"] and np.isnan(zl.loc[2, "lead_time_min"])


# ------------------------------------------------------------------ frame metrics
def test_frame_precision_and_recall_by_size():
    # one VIIRS pass: a 5-pixel fire at the origin, a single-pixel fire 40 km east
    v = viirs([(0, 0, 0), (0, 0.3, 0), (1, 0.6, 0), (1, 0, 0.3), (2, 0.3, 0.3), (2, 40, 0)])
    # MTG at the pass: one detection on the big fire, one far from any VIIRS fire
    det = mtg([(0, 0.2, 0.2), (0, 80, 80)])
    t = fa.frame_metrics(det, v, reps=200).set_index("metric")
    assert t.loc["precision (detections with VIIRS nearby)", "value"] == pytest.approx(0.5)
    assert t.loc["recall, fire size 4-6", "value"] == 1.0 and t.loc["recall, fire size 1 VIIRS pixel", "value"] == 0.0
    assert t.loc["recall, all fire clusters", "n"] == 2


def test_known_pass_without_viirs_detections_counts_as_false_alarms():
    v = viirs([(0, 0, 0)])
    det = mtg([(0, 0, 0), (600, 5, 5)])                                # second detection during a pass with no fires
    t = fa.frame_metrics(det, v, pass_times=[(T0, T0), (T0 + pd.Timedelta(minutes=600), T0 + pd.Timedelta(minutes=600))],
                         reps=200).set_index("metric")
    assert t.loc["precision (detections with VIIRS nearby)", "value"] == pytest.approx(0.5)


# ------------------------------------------------------------------ statistics, loading, export
def test_bootstrap_interval_contains_the_estimate():
    p, lo, hi = fa.bootstrap_prop(30, 100)
    assert lo <= p <= hi and p == 0.3
    m, lo, hi = fa.bootstrap_median([10, 20, 30, 40, 50])
    assert lo <= m <= hi and m == 30


def test_load_csv_with_firms_style_time(tmp_path):
    f = tmp_path / "mtg.csv"
    f.write_text("latitude,longitude,acq_date,acq_time,frp\n36.7,4.3,2025-08-10,130,12.5\n36.7,4.3,2025-08-10,1340,8\n")
    d = fa.load_detections_csv(f)
    assert list(d.time.dt.hour) == [1, 13] and d.time.dt.tz is not None and d.pixel_km.isna().all()


def test_export_json_roundtrip(tmp_path):
    import json
    alerts = fa.match_alerts(fa.build_alerts(mtg([(0, 0, 0), (10, 0, 0)])), viirs([(180, 0, 0)]))
    fa.export_alerts_json(alerts, tmp_path / "alerts.json")
    rec = json.loads((tmp_path / "alerts.json").read_text())[0]
    assert rec["status"] == "confirmed" and rec["lead_time_min"] == 180.0 and rec["zone_id"] is None
    assert rec["first_detected"] == "2025-08-10T10:00:00Z"

_tests = {k: v for k, v in list(globals().items()) if k.startswith("test_") and callable(v)}
_failed = []
for _name, _fn in _tests.items():
    try:
        _fn(pathlib.Path(tempfile.mkdtemp())) if "tmp_path" in inspect.signature(_fn).parameters else _fn()
    except Exception as exc:
        _failed.append((_name, repr(exc)))
print(f"{len(_tests) - len(_failed)} of {len(_tests)} tests passed", *_failed, sep="\n")

## 9. Run on data
Set `MTG_CSV` to a CSV of geostationary detections (needs latitude, longitude and a time column). While it is `None`, the cell
uses **fake detections**, and only the alert and lead-time numbers mean anything.

In [ ]:
MTG_CSV = None            # e.g. "mtg_detections.csv"

det = load_detections_csv(MTG_CSV) if MTG_CSV else make_synthetic_mtg(df, zones)
print("DATA:", "real detections" if MTG_CSV else "SYNTHETIC DRY RUN", "|", len(det), "detections")

alerts = match_alerts(build_alerts(det), df, hotspot_zone_map(zones))
zl = zone_lead_times(zones, alerts)
print(len(alerts), "alerts:", alerts.status.value_counts().to_dict(), "| persistent:", int(alerts.persistent.sum()))
display(zl.round(1))
for k, v in summarize(alerts, zl).items():
    print(f"{k}: {tuple(round(float(x), 2) for x in v) if isinstance(v, tuple) else v}")

In [ ]:
if MTG_CSV:                       # recall needs real detections; on fake data it is not meaningful
    display(frame_metrics(det, df).round(2))

import pathlib
out = pathlib.Path("data") / EVENT; out.mkdir(parents=True, exist_ok=True)
export_alerts_json(alerts, out / "alerts.json")
print("wrote", out / "alerts.json")